# Store Transactions Data Generator

Generates synthetic data for uc1_raw_store_transactions.

Run cells in order. Each cell is independent and can be
re-run to regenerate a specific batch without affecting others.

In [0]:
import uuid
import random
import math
from datetime import date, datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType
)

# ── Configuration ──────────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")
TABLE_NAME  = "uc1_raw_store_transactions"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

TODAY = date.today()

# ── Reference Data ─────────────────────────────────────────────────────────

STORES    = [f"ST{str(i).zfill(3)}" for i in range(1, 26)]
REGIONS   = {
    "ST001": "NORTH",  "ST002": "NORTH",  "ST003": "NORTH",
    "ST004": "NORTH",  "ST005": "NORTH",  "ST006": "SOUTH",
    "ST007": "SOUTH",  "ST008": "SOUTH",  "ST009": "SOUTH",
    "ST010": "SOUTH",  "ST011": "EAST",   "ST012": "EAST",
    "ST013": "EAST",   "ST014": "EAST",   "ST015": "EAST",
    "ST016": "WEST",   "ST017": "WEST",   "ST018": "WEST",
    "ST019": "WEST",   "ST020": "WEST",   "ST021": "CENTRAL",
    "ST022": "CENTRAL","ST023": "CENTRAL","ST024": "CENTRAL",
    "ST025": "CENTRAL"
}
TERMINALS = [f"T{str(i).zfill(3)}" for i in range(1, 6)]
CASHIERS  = [f"C{str(i).zfill(4)}" for i in range(1001, 1201)]

TRANSACTION_TYPES  = ["SALE", "RETURN", "EXCHANGE", "VOID"]
TRANSACTION_WEIGHTS = [0.82, 0.10, 0.05, 0.03]

PAYMENT_METHODS  = ["CASH", "CARD", "VOUCHER", "MOBILE", "GIFT_CARD"]
PAYMENT_WEIGHTS  = [0.20, 0.55, 0.08, 0.12, 0.05]

CHANNELS = ["IN_STORE", "CLICK_AND_COLLECT"]
CHANNEL_WEIGHTS = [0.88, 0.12]

# ── Volume Parameters ──────────────────────────────────────────────────────

DAILY_MIN = 500
DAILY_MAX = 800

# ── Helper Functions ───────────────────────────────────────────────────────

def random_transaction_timestamp(business_date: date) -> datetime:
    """
    Generates a realistic transaction timestamp within business hours.
    Store hours: 08:00 to 22:00
    """
    hour   = random.randint(8, 21)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    return datetime(
        business_date.year,
        business_date.month,
        business_date.day,
        hour, minute, second
    )


def random_amounts(txn_type: str):
    """
    Generates financially consistent gross, discount, tax, and net amounts.
    Returns: (gross, discount, tax, net)
    """
    if txn_type == "VOID":
        return 0.0, 0.0, 0.0, 0.0

    if txn_type == "RETURN":
        gross    = round(-random.uniform(2.0, 200.0), 2)
        discount = 0.0
        tax      = round(gross * 0.08, 2)
        net      = round(gross + tax, 2)
        return gross, discount, tax, net

    # SALE or EXCHANGE
    gross    = round(random.uniform(2.0, 500.0), 2)

    # 30% of transactions have a discount
    if random.random() < 0.30:
        discount = round(gross * random.uniform(0.05, 0.25), 2)
    else:
        discount = 0.0

    tax = round((gross - discount) * 0.08, 2)
    net = round(gross - discount + tax, 2)
    return gross, discount, tax, net


def generate_transaction(
    business_date: date,
    store_id: str = None,
    cashier_id: str = None,
    payment_method_override: str = None,
    txn_type_override: str = None,
    force_null_net_amount: bool = False,
    force_null_store_id: bool = False,
    force_cross_col_error: bool = False,
    gross_override: float = None
) -> dict:
    """
    Generates a single clean or modified transaction record.
    """
    store    = store_id or random.choice(STORES)
    txn_type = txn_type_override or random.choices(
        TRANSACTION_TYPES, weights=TRANSACTION_WEIGHTS
    )[0]
    payment  = payment_method_override or random.choices(
        PAYMENT_METHODS, weights=PAYMENT_WEIGHTS
    )[0]
    channel  = random.choices(CHANNELS, weights=CHANNEL_WEIGHTS)[0]
    cashier  = cashier_id or random.choice(CASHIERS)

    gross, discount, tax, net = random_amounts(txn_type)

    if gross_override is not None:
        gross = gross_override

    # Cross column error: make net_amount deliberately wrong
    if force_cross_col_error and txn_type not in ("VOID", "RETURN"):
        net = round(net + random.uniform(5.0, 50.0), 2)

    item_count = random.randint(1, 20) if txn_type != "VOID" else 0
    qty_total  = random.randint(item_count, item_count * 3) if item_count > 0 else 0

    return {
        "transaction_id"        : str(uuid.uuid4()),
        "transaction_date"      : business_date,
        "transaction_timestamp" : random_transaction_timestamp(business_date),
        "store_id"              : None if force_null_store_id else store,
        "store_region"          : REGIONS.get(store, "UNKNOWN"),
        "terminal_id"           : random.choice(TERMINALS),
        "cashier_id"            : cashier,
        "customer_id"           : str(uuid.uuid4()) if random.random() > 0.35 else None,
        "loyalty_card_number"   : f"LC{random.randint(1000000, 9999999)}"
                                  if random.random() > 0.40 else None,
        "transaction_type"      : txn_type,
        "payment_method"        : payment,
        "channel"               : channel,
        "gross_amount"          : gross,
        "discount_amount"       : discount,
        "tax_amount"            : tax,
        "net_amount"            : None if force_null_net_amount else net,
        "currency_code"         : "GBP",
        "item_count"            : item_count,
        "quantity_total"        : qty_total,
        "source_file_name"      : f"transactions_{business_date}.csv",
        "ingestion_timestamp"   : datetime.now(),
        "_rescued_data"         : None
    }


from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType
)

# ── Explicit Schema for raw_store_transactions ─────────────────────────────

TXN_SCHEMA = StructType([
    StructField("transaction_id",        StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), True),
    StructField("store_id",              StringType(),    True),
    StructField("store_region",          StringType(),    True),
    StructField("terminal_id",           StringType(),    True),
    StructField("cashier_id",            StringType(),    True),
    StructField("customer_id",           StringType(),    True),
    StructField("loyalty_card_number",   StringType(),    True),
    StructField("transaction_type",      StringType(),    True),
    StructField("payment_method",        StringType(),    True),
    StructField("channel",               StringType(),    True),
    StructField("gross_amount",          DoubleType(),    True),
    StructField("discount_amount",       DoubleType(),    True),
    StructField("tax_amount",            DoubleType(),    True),
    StructField("net_amount",            DoubleType(),    True),
    StructField("currency_code",         StringType(),    True),
    StructField("item_count",            IntegerType(),   True),
    StructField("quantity_total",        IntegerType(),   True),
    StructField("source_file_name",      StringType(),    True),
    StructField("ingestion_timestamp",   TimestampType(), True),
    StructField("_rescued_data",         StringType(),    True),
])


def write_batch(records: list, description: str):
    """
    Writes a list of records to the target Delta table
    using an explicit schema to avoid type inference errors
    when nullable fields contain None across all records.
    """
    # Convert list of dicts to list of tuples in schema column order
    column_order = [field.name for field in TXN_SCHEMA.fields]

    rows = [
        tuple(r.get(col) for col in column_order)
        for r in records
    ]

    df = spark.createDataFrame(rows, schema=TXN_SCHEMA)

    df.write \
      .mode("append") \
      .partitionBy("transaction_date") \
      .saveAsTable(TARGET_TABLE)

    print(f"  ✓ {description}")
    print(f"    Records written : {len(records)}")
    print(f"    Business date   : {records[0]['transaction_date']}")
    print(f"    Target table    : {TARGET_TABLE}")

print("Configuration loaded.")
print(f"Target table: {TARGET_TABLE}")

## Batch 1: 30 Days Clean Historical Data

Loads 30 days of clean transaction data.
This is the baseline period the agent will learn from.

Volume: 500-800 records per day
Quality: All checks should PASS

In [0]:
print("Generating 30 days of clean historical data...")
print()

historical_dates = [
    TODAY + timedelta(days=offset)
    for offset in range(-37, -7)
]


total_records = 0

for business_date in historical_dates:
    daily_volume = random.randint(DAILY_MIN, DAILY_MAX)
    records = [
        generate_transaction(business_date)
        for _ in range(daily_volume)
    ]
    write_batch(records, f"Historical clean data: {business_date}")
    total_records += daily_volume

print()
print(f"Batch 1 complete. Total records: {total_records}")
print(f"Date range: {historical_dates[0]} to {historical_dates[-1]}")
print("Expected agent result: All checks PASS (baseline data)")



## Batch 2: Clean Live Day

First live day after baseline period. All checks should pass.
Used to verify the agent correctly handles a clean load
before we start injecting quality issues.

In [0]:
clean_live_date = TODAY + timedelta(days=-6)
daily_volume    = random.randint(DAILY_MIN, DAILY_MAX)

print(f"Generating clean live batch for: {clean_live_date}")

records = [
    generate_transaction(clean_live_date)
    for _ in range(daily_volume)
]

write_batch(records, f"Clean live batch: {clean_live_date}")

print()
print("Batch 2 complete.")
print("Expected agent result: All checks PASS")

## Batch 3: Volume Drop

Injects a 75% volume drop below the baseline mean.
Simulates a partial batch job failure or truncated source file.

Normal volume: 500-800 rows
Injected volume: ~150 rows (approximately 75% below baseline mean)

Expected agent result:
  VOLUME_ANOMALY → CRITICAL (exceeds 40% critical threshold)

In [0]:
volume_drop_date   = TODAY + timedelta(days=-5)
injected_volume    = 150   # ~75% below normal mean of ~650

print(f"Generating volume drop batch for: {volume_drop_date}")
print(f"Normal range:    {DAILY_MIN} to {DAILY_MAX}")
print(f"Injected volume: {injected_volume} rows")
print(f"Drop magnitude:  ~{round((1 - injected_volume / 650) * 100)}% below baseline mean")

records = [
    generate_transaction(volume_drop_date)
    for _ in range(injected_volume)
]

write_batch(records, f"Volume drop batch: {volume_drop_date}")

print()
print("Batch 3 complete.")
print("Expected agent result:")
print("  VOLUME_ANOMALY → CRITICAL")
print(f"  Observed: {injected_volume} rows")
print(f"  Baseline mean: ~650 rows")
print(f"  Deviation: ~77% below baseline")

## Batch 4: Null Spike in net_amount

Injects a 22% null rate in the net_amount column.
Simulates a POS system version change that stopped
populating net_amount for a subset of terminal types.

Normal null rate for net_amount: ~0.1%
Injected null rate: 22%

Expected agent result:
  NULL_SPIKE on net_amount → CRITICAL (exceeds 5% critical threshold)

In [0]:
null_spike_date  = TODAY + timedelta(days=-4)
daily_volume     = random.randint(DAILY_MIN, DAILY_MAX)
null_spike_rate  = 0.22    # 22% of records will have null net_amount

null_count   = int(daily_volume * null_spike_rate)
normal_count = daily_volume - null_count

print(f"Generating null spike batch for: {null_spike_date}")
print(f"Total records:      {daily_volume}")
print(f"Null net_amount:    {null_count} records ({null_spike_rate*100:.0f}%)")
print(f"Normal records:     {normal_count} records")

records = []

# Normal records
for _ in range(normal_count):
    records.append(generate_transaction(null_spike_date))

# Records with null net_amount
for _ in range(null_count):
    records.append(
        generate_transaction(
            null_spike_date,
            force_null_net_amount=True
        )
    )

random.shuffle(records)

write_batch(records, f"Null spike batch: {null_spike_date}")

print()
print("Batch 4 complete.")
print("Expected agent result:")
print("  NULL_SPIKE on net_amount → CRITICAL")
print(f"  Observed null rate: {null_spike_rate*100:.0f}%")
print(f"  Baseline null rate: ~0.1%")
print(f"  Warning threshold:  1.0%")
print(f"  Critical threshold: 5.0%")

## Batch 5: Categorical Domain Violation in payment_method

Introduces two new payment method values not in the domain list:
  BNPL  (Buy Now Pay Later) in 8% of records
  CRYPTO in 0.5% of records

Simulates a new payment type launched at source POS
without the data team being notified.

Expected agent result:
  CATEGORICAL_DOMAIN on payment_method → WARNING
  (8.5% invalid values, exceeds 0.5% warning threshold)

In [0]:
categorical_date = TODAY + timedelta(days=-3)
daily_volume     = random.randint(DAILY_MIN, DAILY_MAX)

bnpl_count   = int(daily_volume * 0.08)    # 8% BNPL
crypto_count = int(daily_volume * 0.005)   # 0.5% CRYPTO
normal_count = daily_volume - bnpl_count - crypto_count

print(f"Generating categorical domain batch for: {categorical_date}")
print(f"Total records:    {daily_volume}")
print(f"Normal payment:   {normal_count}")
print(f"BNPL (invalid):   {bnpl_count} ({bnpl_count/daily_volume*100:.1f}%)")
print(f"CRYPTO (invalid): {crypto_count} ({crypto_count/daily_volume*100:.1f}%)")

records = []

# Normal records with valid payment methods
for _ in range(normal_count):
    records.append(generate_transaction(categorical_date))

# Records with BNPL payment method (not in domain list)
for _ in range(bnpl_count):
    records.append(
        generate_transaction(
            categorical_date,
            payment_method_override="BNPL"
        )
    )

# Records with CRYPTO payment method (not in domain list)
for _ in range(crypto_count):
    records.append(
        generate_transaction(
            categorical_date,
            payment_method_override="CRYPTO"
        )
    )

random.shuffle(records)

write_batch(records, f"Categorical domain batch: {categorical_date}")

print()
print("Batch 5 complete.")
print("Expected agent result:")
print("  CATEGORICAL_DOMAIN on payment_method → WARNING")
print(f"  Invalid values: BNPL ({bnpl_count/daily_volume*100:.1f}%), "
      f"CRYPTO ({crypto_count/daily_volume*100:.1f}%)")
print(f"  Total invalid rate: {(bnpl_count+crypto_count)/daily_volume*100:.1f}%")
print(f"  Warning threshold: 0.5%")
print(f"  Critical threshold: 2.0%")

## Batch 6: Cross-Column Consistency Violation

Breaks the rule: gross_amount - discount_amount + tax_amount = net_amount
for 12% of records by introducing a rounding error in net_amount.

Simulates a calculation change in the source billing system
where new rounding logic was applied inconsistently.

Expected agent result:
  CROSS_COLUMN_CONSISTENCY on net_amount → WARNING
  (12% violation rate, exceeds 1% warning threshold)

In [0]:
cross_col_date   = TODAY + timedelta(days=-2)
daily_volume     = random.randint(DAILY_MIN, DAILY_MAX)
violation_rate   = 0.12    # 12% of records will have inconsistent amounts

violation_count  = int(daily_volume * violation_rate)
normal_count     = daily_volume - violation_count

print(f"Generating cross-column consistency batch for: {cross_col_date}")
print(f"Total records:      {daily_volume}")
print(f"Consistent records: {normal_count}")
print(f"Violations:         {violation_count} ({violation_rate*100:.0f}%)")
print(f"Rule broken:        gross - discount + tax != net_amount")

records = []

# Normal consistent records
for _ in range(normal_count):
    records.append(generate_transaction(cross_col_date))

# Records with cross-column inconsistency
for _ in range(violation_count):
    records.append(
        generate_transaction(
            cross_col_date,
            force_cross_col_error=True
        )
    )

random.shuffle(records)

write_batch(records, f"Cross-column consistency batch: {cross_col_date}")

print()
print("Batch 6 complete.")
print("Expected agent result:")
print("  CROSS_COLUMN_CONSISTENCY on net_amount → WARNING")
print(f"  Violation rate: {violation_rate*100:.0f}%")
print(f"  Warning threshold: 1.0%")
print(f"  Critical threshold: 5.0%")

## Batch 7: Combined Issues

Three simultaneous WARNING-level issues:
  1. Volume 35% below normal
  2. store_id null rate at 5%
  3. New transaction_type value LAYAWAY in 3% of records

No single issue crosses CRITICAL alone but the combined
pattern should trigger AI confidence threshold.

Expected agent result:
  VOLUME_ANOMALY → WARNING
  NULL_SPIKE on store_id → WARNING
  CATEGORICAL_DOMAIN on transaction_type → WARNING
  Overall → WARNING with AI invocation triggered

In [0]:
combined_date    = TODAY + timedelta(days=-1)
reduced_volume   = int(650 * 0.65)    # 35% below baseline mean of 650
null_store_rate  = 0.05               # 5% null store_id
layaway_rate     = 0.03               # 3% LAYAWAY transaction type

null_store_count = int(reduced_volume * null_store_rate)
layaway_count    = int(reduced_volume * layaway_rate)
normal_count     = reduced_volume - null_store_count - layaway_count

print(f"Generating combined issues batch for: {combined_date}")
print(f"Normal volume:        ~650")
print(f"Injected volume:      {reduced_volume} (~35% below)")
print(f"Null store_id:        {null_store_count} records ({null_store_rate*100:.0f}%)")
print(f"LAYAWAY type:         {layaway_count} records ({layaway_rate*100:.0f}%)")
print(f"Normal records:       {normal_count}")

records = []

# Normal records
for _ in range(normal_count):
    records.append(generate_transaction(combined_date))

# Records with null store_id
for _ in range(null_store_count):
    records.append(
        generate_transaction(
            combined_date,
            force_null_store_id=True
        )
    )

# Records with invalid transaction_type LAYAWAY
for _ in range(layaway_count):
    records.append(
        generate_transaction(
            combined_date,
            txn_type_override="LAYAWAY"
        )
    )

random.shuffle(records)

write_batch(records, f"Combined issues batch: {combined_date}")

print()
print("Batch 7 complete.")
print("Expected agent result:")
print("  VOLUME_ANOMALY → WARNING (35% below baseline)")
print(f"  NULL_SPIKE on store_id → WARNING ({null_store_rate*100:.0f}% null rate)")
print(f"  CATEGORICAL_DOMAIN on transaction_type → WARNING (LAYAWAY not in domain)")
print("  Overall → WARNING")
print("  AI invocation → TRIGGERED (3 simultaneous check failures)")

## Verification: Confirm All Batches Were Written Correctly

In [0]:
print("=" * 65)
print("STORE TRANSACTIONS — DATA GENERATION VERIFICATION")
print("=" * 65)

# Row count by date
print("\n1. Row count by transaction_date:\n")
spark.sql(f"""
    SELECT
        transaction_date,
        COUNT(*)                                            AS row_count,
        ROUND(SUM(CASE WHEN net_amount IS NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_null_net_amount,
        ROUND(SUM(CASE WHEN store_id IS NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS pct_null_store_id
    FROM {TARGET_TABLE}
    GROUP BY transaction_date
    ORDER BY transaction_date
""").show(50, truncate=False)

# Payment method distribution for controlled batch
print("\n2. Payment method distribution (categorical batch):\n")
spark.sql(f"""
    SELECT
        payment_method,
        COUNT(*) AS record_count
    FROM {TARGET_TABLE}
    WHERE transaction_date = '{TODAY + timedelta(days=-3)}'
    GROUP BY payment_method
    ORDER BY record_count DESC
""").show(truncate=False)

# Cross-column consistency check
print("\n3. Cross-column consistency violations (cross-column batch):\n")
spark.sql(f"""
    SELECT
        COUNT(*)                                         AS total_records,
        SUM(CASE
                WHEN ABS(gross_amount
                         - COALESCE(discount_amount, 0)
                         + COALESCE(tax_amount, 0)
                         - net_amount) > 0.01
                THEN 1 ELSE 0
            END)                                         AS violations,
        ROUND(SUM(CASE
                      WHEN ABS(gross_amount
                               - COALESCE(discount_amount, 0)
                               + COALESCE(tax_amount, 0)
                               - net_amount) > 0.01
                      THEN 1 ELSE 0
                  END) / COUNT(*) * 100, 2)              AS violation_pct
    FROM {TARGET_TABLE}
    WHERE transaction_date = '{TODAY + timedelta(days=-2)}'
      AND transaction_type NOT IN ('VOID', 'RETURN')
""").show(truncate=False)

# Transaction type distribution for combined batch
print("\n4. Transaction types in combined batch:\n")
spark.sql(f"""
    SELECT
        transaction_type,
        COUNT(*) AS record_count
    FROM {TARGET_TABLE}
    WHERE transaction_date = '{TODAY + timedelta(days=-1)}'
    GROUP BY transaction_type
    ORDER BY record_count DESC
""").show(truncate=False)

# Overall summary
print("\n5. Overall data generation summary:\n")
spark.sql(f"""
    SELECT
        MIN(transaction_date)   AS earliest_date,
        MAX(transaction_date)   AS latest_date,
        COUNT(DISTINCT transaction_date) AS distinct_dates,
        COUNT(*)                AS total_records,
        ROUND(AVG(row_count), 0) AS avg_daily_rows
    FROM (
        SELECT
            transaction_date,
            COUNT(*) AS row_count
        FROM {TARGET_TABLE}
        GROUP BY transaction_date
    )
""").show(truncate=False)

print("Verification complete.")